# Model Inference — Kaggle Submission

Loads the **best** model from the MLflow **Model Registry**, predicts on the **raw** test set, and writes a Kaggle submission. No manual preprocessing: the registered Pipeline handles merge + feature engineering internally.

In [ ]:
# === Bootstrap: make the shared `src` package importable (local + Kaggle) ===
import sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/sophyrise/walmart-sales-forecasting.git"

if Path("/kaggle/input").exists():
    # On Kaggle: clone the repo to pull in the shared src/ package.
    dst = Path("/kaggle/working/walmart-sales-forecasting")
    if not dst.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dst)], check=True)
    sys.path.insert(0, str(dst))
else:
    # Local: notebooks/ sits one level under the repo root.
    sys.path.insert(0, str(Path.cwd().parent))

from src import config
print("On Kaggle:", config.ON_KAGGLE, "| raw data dir:", config.RAW_DIR)

In [ ]:
import mlflow
from src import data_loader as dl
from src.submission import make_submission
from src.tracking import init_tracking

init_tracking()
MODEL_NAME = 'walmart_best_model'   # registered name
STAGE_OR_VERSION = 'latest'          # or a specific version / alias

## Load best model from the Model Registry

In [ ]:
model_uri = f'models:/{MODEL_NAME}/{STAGE_OR_VERSION}'
model = mlflow.sklearn.load_model(model_uri)
print('loaded', model_uri)

## Predict on the raw test set

In [ ]:
raw_test = dl.load_raw('test')
preds = model.predict(raw_test)
sub = make_submission(raw_test, preds)   # clips negatives, writes submission.csv
sub.head()

## Submit to Kaggle (optional)
Requires Kaggle credentials (KAGGLE_USERNAME / KAGGLE_KEY).

In [ ]:
# !kaggle competitions submit -c walmart-recruiting-store-sales-forecasting \
#     -f submission.csv -m 'best model from registry'